In [1]:
import pandas as pd
df = pd.read_csv('../data/loan_raw.csv.gz')
print(df.shape)

C:\Users\91703\AppData\Local\Temp\ipykernel_5020\2687718265.py:2: DtypeWarning: Columns (0,19,49,59,118,129,130,131,134,135,136,139,145,146,147) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/loan_raw.csv.gz')


(2260701, 151)


In [2]:
df['loan_status'].value_counts()

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

In [3]:
default_statuses = ['Charged Off', 'Does not meet the credit policy. Status:Charged Off']
paid_statuses = ['Fully Paid', 'Does not meet the credit policy. Status:Fully Paid']

df_clean = df[df['loan_status'].isin(default_statuses + paid_statuses)].copy()
df_clean['default'] = df_clean['loan_status'].isin(default_statuses).astype(int)

print(df_clean.shape)
df_clean['default'].value_counts()

(1348059, 152)


default
0    1078739
1     269320
Name: count, dtype: int64

In [4]:
missing = df_clean.isnull().sum()
missing_pct = (missing / len(df_clean) * 100).round(2)
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_summary = missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print(missing_summary.shape[0], "columns have missing values")
missing_summary.head(30)

113 columns have missing values


,missing_count,missing_pct
member_id,1348059,100.00
next_pymnt_d,1345310,99.80
orig_projected_additional_accrued_interest,1344300,99.72
hardship_loan_status,1342305,99.57
hardship_dpd,1342305,99.57
hardship_start_date,1342305,99.57
hardship_end_date,1342305,99.57
hardship_status,1342305,99.57
hardship_type,1342305,99.57
hardship_amount,1342305,99.57


In [5]:
threshold = 40  # drop columns missing more than 40%
cols_to_drop = missing_summary[missing_summary['missing_pct'] > threshold].index.tolist()
df_clean = df_clean.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns")
print(df_clean.shape)

Dropped 58 columns
(1348059, 94)


In [6]:
remaining_missing = df_clean.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0].sort_values(ascending=False)
remaining_pct = (remaining_missing / len(df_clean) * 100).round(2)
pd.DataFrame({'missing_count': remaining_missing, 'missing_pct': remaining_pct})

,missing_count,missing_pct
mths_since_recent_inq,176820,13.12
num_tl_120dpd_2m,120150,8.91
mo_sin_old_il_acct,108324,8.04
emp_title,85944,6.38
emp_length,78545,5.83
pct_tl_nvr_dlq,70430,5.22
avg_cur_bal,70298,5.21
num_rev_accts,70277,5.21
mo_sin_old_rev_tl_op,70277,5.21
mo_sin_rcnt_rev_tl_op,70277,5.21


In [7]:
# Group 1: drop low-value text columns
df_clean = df_clean.drop(columns=['emp_title', 'title'])

# Group 2: median imputation for numeric columns
numeric_impute_cols = ['mths_since_recent_inq','num_tl_120dpd_2m','mo_sin_old_il_acct',
    'pct_tl_nvr_dlq','avg_cur_bal','num_rev_accts','mo_sin_old_rev_tl_op','mo_sin_rcnt_rev_tl_op',
    'mo_sin_rcnt_tl','num_tl_90g_dpd_24m','num_tl_30dpd','num_bc_tl','num_il_tl','num_actv_rev_tl',
    'num_accts_ever_120_pd','num_actv_bc_tl','num_rev_tl_bal_gt_0','total_rev_hi_lim','tot_cur_bal',
    'tot_coll_amt','total_il_high_credit_limit','num_tl_op_past_12m','tot_hi_cred_lim','num_op_rev_tl',
    'bc_util','percent_bc_gt_75','bc_open_to_buy','mths_since_recent_bc','num_bc_sats','num_sats',
    'total_bal_ex_mort','mort_acc','total_bc_limit','acc_open_past_24mths']
for col in numeric_impute_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Group 3: mode imputation for emp_length
df_clean['emp_length'] = df_clean['emp_length'].fillna(df_clean['emp_length'].mode()[0])

# Group 4: drop rows with remaining tiny missingness
df_clean = df_clean.dropna()

print(df_clean.shape)
print(df_clean.isnull().sum().sum(), "total missing values left")

(1343086, 92)
0 total missing values left
